In [2]:
# ============================================================
# 8. EVALUATE RECOMMENDATION RELEVANCE
#    AND FORECAST ACCURACY
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# STEP 1: LOAD YOUR ACTUAL DATASET
# ============================================================

df = pd.read_csv("/content/smartagri_crop_yield_dataset.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# STEP 2: PREPARE DATA
# ============================================================

# Convert Crop and Soil_Type into numerical columns
data = pd.get_dummies(
    df,
    columns=["Crop", "Soil_Type"],
    dtype=int
)

# Input features
X = data.drop("Yield", axis=1)

# Target
y = data["Yield"]


# ============================================================
# STEP 3: SPLIT DATA
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))


# ============================================================
# STEP 4: TRAIN YIELD FORECAST MODEL
# ============================================================

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("\nYield forecasting model trained successfully!")


# ============================================================
# STEP 5: PREDICT YIELD
# ============================================================

y_pred = model.predict(X_test)


# ============================================================
# STEP 6: CALCULATE FORECAST ACCURACY
# ============================================================

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)


print("\n================================================")
print("          YIELD FORECAST ACCURACY")
print("================================================")

print("MAE  :", round(mae, 2))
print("RMSE :", round(rmse, 2))
print("R²   :", round(r2, 2))


# ============================================================
# STEP 7: CREATE FORECAST RESULT TABLE
# ============================================================

forecast_results = pd.DataFrame({
    "Actual Yield": y_test.values,
    "Predicted Yield": y_pred
})

forecast_results["Difference"] = abs(
    forecast_results["Actual Yield"]
    - forecast_results["Predicted Yield"]
)

print("\n================================================")
print("          SAMPLE FORECAST RESULTS")
print("================================================")

print(
    forecast_results.head(10).round(2)
)


# ============================================================
# STEP 8: RECOMMENDATION FUNCTION
# ============================================================

def get_recommendation(row):

    recommendations = []

    # Nitrogen condition
    if row["Nitrogen"] < 50:
        recommendations.append(
            "Check nitrogen requirement"
        )

    # Rainfall condition
    if row["Rainfall"] < 500:
        recommendations.append(
            "Consider additional irrigation"
        )

    # Humidity condition
    if row["Humidity"] > 80:
        recommendations.append(
            "Monitor fungal disease risk"
        )

    # Soil pH condition
    if row["Soil_pH"] < 6:
        recommendations.append(
            "Check soil pH correction"
        )

    # Normal condition
    if len(recommendations) == 0:
        recommendations.append(
            "Continue regular crop management"
        )

    return recommendations


# ============================================================
# STEP 9: TEST RECOMMENDATIONS
# ============================================================

test_data = df.head(20)

recommendation_results = []

for index, row in test_data.iterrows():

    recommendations = get_recommendation(row)

    recommendation_results.append({
        "Crop": row["Crop"],
        "Recommendations": "; ".join(recommendations)
    })


recommendation_df = pd.DataFrame(
    recommendation_results
)


# ============================================================
# STEP 10: DISPLAY RECOMMENDATIONS
# ============================================================

print("\n================================================")
print("        RECOMMENDATION RELEVANCE TEST")
print("================================================")

print(
    recommendation_df.to_string(index=False)
)


# ============================================================
# STEP 11: CHECK RELEVANCE
# ============================================================

relevant_count = 0

for recommendations in recommendation_df["Recommendations"]:

    if recommendations != "":
        relevant_count += 1


total_cases = len(recommendation_df)

relevance_percentage = (
    relevant_count / total_cases
) * 100


print("\n================================================")
print("          RECOMMENDATION EVALUATION")
print("================================================")

print("Total cases tested:", total_cases)

print(
    "Relevant recommendations:",
    relevant_count
)

print(
    "Recommendation relevance:",
    round(relevance_percentage, 2),
    "%"
)


# ============================================================
# STEP 12: FINAL SUMMARY
# ============================================================

print("\n================================================")
print("                 FINAL SUMMARY")
print("================================================")

print("Dataset records:", len(df))

print(
    "Yield Forecast MAE:",
    round(mae, 2)
)

print(
    "Yield Forecast RMSE:",
    round(rmse, 2)
)

print(
    "Yield Forecast R²:",
    round(r2, 2)
)

print(
    "Recommendation Relevance:",
    round(relevance_percentage, 2),
    "%"
)

print("\nEvaluation completed successfully!")

Dataset loaded successfully!
Dataset Shape: (600, 10)

Columns:
['Crop', 'Soil_Type', 'Soil_pH', 'Nitrogen', 'Phosphorus', 'Potassium', 'Temperature', 'Rainfall', 'Humidity', 'Yield']

Training records: 480
Testing records: 120

Yield forecasting model trained successfully!

          YIELD FORECAST ACCURACY
MAE  : 0.98
RMSE : 2.39
R²   : 0.99

          SAMPLE FORECAST RESULTS
   Actual Yield  Predicted Yield  Difference
0          2.00             2.13        0.13
1         68.22            70.17        1.95
2         55.09            66.58       11.49
3          3.71             2.76        0.95
4          4.36             4.05        0.31
5          4.64             4.49        0.15
6         60.92            64.43        3.51
7          1.78             1.72        0.06
8          3.69             3.60        0.09
9          3.24             3.28        0.04

        RECOMMENDATION RELEVANCE TEST
     Crop                                                                      Recomm